# 🇧🇩 Context-Aware Bangla Text Analyzer: Interactive Model Tester
**Course:** CSE 4121 (Natural Language Processing Sessional)  
**Supervised by:** Md. Shawon Sir  
**Project Members:** Md. Tariful Islam Jony (2107119) & Siyam Khan (2107120)  

---
### 🎯 How to Use This Notebook:
1. Make sure your kernel in the top-right corner is set to **`Python (.venv)`**.
2. Run **Cell 1** (Setup & Imports) and **Cell 2** (Load Models) once.
3. In **Cell 4**, type any Bangla sentence in `my_sentence` and press `Shift + Enter` to see live comparative predictions!


In [1]:
# Cell 1: Environment Setup & Importers
import os
import sys

# Auto-link project virtual environment packages if running on another kernel
venv_site = r'D:\D Drive\CSE 4-1\CSE 4121\Lab\.venv\Lib\site-packages'
if os.path.exists(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)

import time
import pandas as pd
from IPython.display import display

# Add src directory to Python path
current_dir = os.path.abspath(os.getcwd())
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from predict_lr import BanglaTextAnalyzerLR
from predict_bilstm import BanglaTextAnalyzerBiLSTM
from predict_bert import BanglaTextAnalyzerBERT

print('✅ Setup complete! Successfully linked src modules & environment.')

✅ Setup complete! Successfully linked src modules & environment.


In [3]:
# Cell 2: Ingest & Initialize All 3 Modeling Paradigms
print('⏳ Loading all 3 models into memory... (Takes ~8-12 seconds)')
t0 = time.time()

lr_model = BanglaTextAnalyzerLR()
bilstm_model = BanglaTextAnalyzerBiLSTM()
bert_model = BanglaTextAnalyzerBERT()

print(f'🎉 All 3 models successfully loaded into memory in {time.time() - t0:.2f} seconds!')

⏳ Loading all 3 models into memory... (Takes ~8-12 seconds)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

🎉 All 3 models successfully loaded into memory in 9.20 seconds!


In [4]:
# Cell 3: Comparison Display Function
def analyze_sentence(text: str):
    """
    Runs inference across all 3 models on the input sentence
    and displays a formatted comparison table.
    """
    if not text or not text.strip():
        print('⚠️ Please enter a non-empty Bangla sentence.')
        return
        
    lr_res = lr_model.analyze(text)
    bi_res = bilstm_model.analyze(text)
    bert_res = bert_model.analyze(text)
    
    rows = []
    tasks = [
        ('Sentiment', 'sentiment'),
        ('Sarcasm', 'sarcasm'),
        ('Hate Speech', 'hate_speech')
    ]
    
    for task_name, task_key in tasks:
        rows.append({
            'Task': task_name,
            'TF-IDF + Logistic Regression': f"{lr_res[task_key]['label']} ({lr_res[task_key]['confidence']}%)",
            'Word2Vec + Stacked BiLSTM': f"{bi_res[task_key]['label']} ({bi_res[task_key]['confidence']}%)",
            'Fine-Tuned BanglaBERT': f"{bert_res[task_key]['label']} ({bert_res[task_key]['confidence']}%)"
        })
        
    rows.append({
        'Task': '⏱️ Latency',
        'TF-IDF + Logistic Regression': f"{lr_res['latency_ms']:.2f} ms",
        'Word2Vec + Stacked BiLSTM': f"{bi_res['latency_ms']:.2f} ms",
        'Fine-Tuned BanglaBERT': f"{bert_res['latency_ms']:.2f} ms"
    })
    
    df = pd.DataFrame(rows)
    
    print(f'\n🔍 Input Text: "{text}"')
    print(f'🧹 Cleaned Text: "{lr_res["cleaned_text"]}"\n')
    
    display(df)

In [5]:
# Cell 4: ✍️ Test ANY Custom Bangla Text Here!
# Change the sentence below to whatever you want, then press Shift + Enter to run:

my_sentence = 'বাহ! কী অসাধারণ service, তিন ঘণ্টা অপেক্ষা করেও কাজ হলো না!'

analyze_sentence(my_sentence)


🔍 Input Text: "বাহ! কী অসাধারণ service, তিন ঘণ্টা অপেক্ষা করেও কাজ হলো না!"
🧹 Cleaned Text: "বাহ! কী অসাধারণ service, তিন ঘণ্টা অপেক্ষা করেও কাজ হলো না!"



,Task,TF-IDF + Logistic Regression,Word2Vec + Stacked BiLSTM,Fine-Tuned BanglaBERT
0,Sentiment,Positive (52.56%),Positive (82.75%),Negative (40.28%)
1,Sarcasm,Sarcastic (57.03%),Non-Sarcastic (58.75%),Non-Sarcastic (59.48%)
2,Hate Speech,Non-Hate (85.05%),Non-Hate (98.31%),Non-Hate (90.87%)
3,⏱️ Latency,46.36 ms,8.93 ms,277.10 ms


In [ ]:
# Cell 5: Detailed Probability Distribution Breakdown
def show_probabilities(text: str):
    lr_res = lr_model.analyze(text)
    bi_res = bilstm_model.analyze(text)
    bert_res = bert_model.analyze(text)
    
    print(f'📊 Detailed Class Probability Breakdown for: "{text}"\n')
    
    for task_key, task_name in [('sentiment', 'Sentiment'), ('sarcasm', 'Sarcasm'), ('hate_speech', 'Hate Speech')]:
        prob_rows = []
        classes = list(lr_res[task_key]['probabilities'].keys())
        for cls in classes:
            prob_rows.append({
                'Class': cls,
                'TF-IDF + LR (%)': lr_res[task_key]['probabilities'].get(cls, 0.0),
                'BiLSTM (%)': bi_res[task_key]['probabilities'].get(cls, 0.0),
                'BanglaBERT (%)': bert_res[task_key]['probabilities'].get(cls, 0.0)
            })
        print(f'--- {task_name} ---')
        display(pd.DataFrame(prob_rows))
        print()

# Test probability breakdown on the current sentence:
show_probabilities(my_sentence)

In [ ]:
# Cell 6: Run Standard Benchmark Suite
benchmark_examples = [
    ('Sarcastic Contrast', 'বাহ! কী অসাধারণ service, তিন ঘণ্টা অপেক্ষা করেও কাজ হলো না!'),
    ('Positive Sentiment', 'বইটা অসম্ভব সুন্দর এবং অনুপ্রেরণামূলক! সবাইকে পড়ার অনুরোধ রইলো।'),
    ('Negative Sentiment', 'একদম বাজে কোয়ালিটি, টাকাটাই নষ্ট হলো, কেউ কিনবেন না।'),
    ('Hate Speech', 'তোদের মতো দেশদ্রোহীদের প্রকাশ্যে ফাঁসি দেওয়া উচিত!'),
    ('Neutral Statement', 'আজকের আবহাওয়াটা বেশ সাধারণ, খুব গরমও না আবার ঠান্ডাও না।')
]

print('🚀 Running Benchmark Evaluation Suite across all 3 paradigms:\n')
for title, text in benchmark_examples:
    print(f'📌 Scenario: {title}')
    analyze_sentence(text)
    print('=' * 80)